# Arena 3D Reconstruction with Gaussian Splatting

Reconstruct a 10-15m arena from 91 photos using COLMAP + 3D Gaussian Splatting,
then compress the model for lightweight local viewing.

## Pipeline Overview (run cells in order)

| Step | Cell | What it does | Time | GPU? |
|------|------|-------------|------|------|
| 1 | **Cell 1** | Mount Google Drive + choose session | ~30s | No |
| 2 | **Cell 2** | Install COLMAP, PyTorch, CUDA extensions | ~3 min | No* |
| 3 | **Cell 3** | Download 91 arena photos from GitHub | ~2 min | No |
| 4 | **Cell 4A-4D** | COLMAP SfM: features, matching, reconstruction, merge | ~25 min | No |
| 5 | **Cell 5** | *OR* download pre-computed COLMAP (30, 34, or 84 images) | ~1 min | No |
| 6 | **Cell 6** | Convert COLMAP to 3DGS format (SIMPLE_RADIAL to PINHOLE) | ~1 min | No |
| 7 | **Cell 7A-7B** | Train 3D Gaussian Splatting (enhanced, gsplat-based) | 7-30 min | **Yes (T4+)** |
| 8 | **Cell 8A** | Export final point cloud PLY | ~1 min | No |
| 9 | **Cell 8B** | Validate PLY for Unity + viewing options | ~1 min | No |
| 10 | **Cell 8C** | Compress model for local decompression viewer | ~1 min | No |

## Session Management

This notebook saves its progress to Google Drive and **restores data on resume**.
If your session disconnects, re-open the notebook and run **Cell 1** - it will
ask if you want to continue from where you left off. All intermediate files
(images, COLMAP database, sparse model) get restored from Drive automatically.

## Outputs

| File | Location | Size |
|------|----------|------|
| Final model (PLY) | `arena_3dgs_pointcloud.ply` (also on Drive) | ~100-500 MB |
| Compressed model (.splat) | `arena_3dgs_compressed.splat` | ~10-50 MB |
| Training output (PLY) | `output/arena_3dgs/arena_3dgs.ply` | ~100-500 MB |
| COLMAP database | `MyDrive/arena_3dgs/database.db` | ~100 MB |
| Sparse model | `MyDrive/arena_3dgs/sparse_model` | ~10 MB |


In [ ]:
#@title === 1. Mount Drive + Session Management ===
import os, sys, json, urllib.request

# On Colab, download session module from GitHub before importing
SCRIPTS_DIR = os.path.join(os.getcwd(), 'scripts') if 'google.colab' not in sys.modules else '/content/scripts'
os.makedirs(SCRIPTS_DIR, exist_ok=True)
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
SESSION_PY = os.path.join(SCRIPTS_DIR, "session.py")
if not os.path.exists(SESSION_PY):
    url = "https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/session.py"
    urllib.request.urlretrieve(url, SESSION_PY)

from scripts.session import Session, get_drive_path

DRIVE_PATH = get_drive_path()
session = Session(DRIVE_PATH)

existing_session = session.load()
has_previous = existing_session["created_at"] is not None

print()
session.print_status(existing_session)

if has_previous:
    choice = input("\nContinue from previous session? [Y/n]: ").strip().lower() or "y"
    if choice == "y":
        print("\n  [session] Continuing previous session. Data will be restored from Drive.")
    else:
        print("\n  [session] Starting fresh.")
        session.reset()
else:
    print("\n  No previous session found. Starting fresh.")
    session.reset()

session.mark_step("drive_mounted")
print(f"\nSession active. Checkpoints in: {DRIVE_PATH}")


In [ ]:
#@title === 2. Install Dependencies (~3 min, idempotent) ===
# Download the pipeline module from GitHub if on Colab
import os, sys, urllib.request
SCRIPTS_DIR = "/content/scripts"
os.makedirs(SCRIPTS_DIR, exist_ok=True)
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)
MODULES = ["session.py", "colab_pipeline.py"]
for mod in MODULES:
    dest = os.path.join(SCRIPTS_DIR, mod)
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/{mod}"
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"Downloaded {mod}")
        except Exception as e:
            print(f"Could not download {mod}: {e}")

from scripts.colab_pipeline import install_dependencies

install_dependencies(session)


---
## Step 3: Images
---


In [ ]:
#@title === 3: Download Images from GitHub (~2 min, idempotent) ===
from scripts.colab_pipeline import download_images

# On resume, images are restored from Drive or re-downloaded
download_images(session)


---
## COLMAP Step 4: Structure from Motion

Three modular cells: (A) features, (B) matching, (C) reconstruction.
Each saves checkpoints to Drive. On resume, data is restored from Drive.

**Alternatively**, skip to Step 5 and download pre-computed camera poses.
---


In [ ]:
#@title === 4A: Feature Extraction (~5 min, idempotent) ===
from scripts.colab_pipeline import run_colmap_features
run_colmap_features(session)


In [ ]:
#@title === 4B: Feature Matching (~5 min, idempotent) ===
from scripts.colab_pipeline import run_colmap_matching
run_colmap_matching(session)


In [ ]:
#@title === 4C: COLMAP Reconstruction (~10 min, idempotent) ===
from scripts.colab_pipeline import run_colmap_reconstruction
run_colmap_reconstruction(session)


In [ ]:
#@title === 4D: Model Merging (~2 min, idempotent) ===
from scripts.colab_pipeline import run_colmap_merge
run_colmap_merge(session)


---
## Step 5: Download Pre-computed COLMAP Data (~1 min)

Skip COLMAP (Step 4) and download pre-computed camera poses.
Choose from three pre-computed models using the dropdown below.
---


In [ ]:
#@title === 5: Download Pre-computed COLMAP Data (~1 min, idempotent) ===

MODEL_CHOICE = "84-image full" #@param ["30-image original", "34-image merged", "84-image full"]

from scripts.colab_pipeline import download_precomputed_colmap, COLMAP_MODELS

info = COLMAP_MODELS[MODEL_CHOICE]
print(f"Selected: {MODEL_CHOICE} ({info['desc']})")
download_precomputed_colmap(session, model_choice=MODEL_CHOICE)


---
## Step 6: Convert to 3DGS Format (~1 min)

Converts SIMPLE_RADIAL camera model to PINHOLE (required by 3DGS text reader).
Rebuilds binary files from text.
---


In [ ]:
#@title === 6: Convert Data to 3DGS Format (~1 min, idempotent) ===
from scripts.colab_pipeline import convert_to_3dgs_format
convert_to_3dgs_format(session)


---
## Step 7: Train 3D Gaussian Splatting

Training uses the enhanced gsplat-based script for faster, more memory-efficient
training without CUDA extension compilation.

**Run order:**
1. Cell 7A: Quick test (3K iters, ~7 min) - verify everything works
2. Cell 7B: Full training (30K iters, ~30 min) - main training run
---


In [ ]:
#@title === 7A: Quick Test (3000 iters, ~7 min) ===
from scripts.colab_pipeline import train_3dgs
train_3dgs(session, iterations=3000, max_gaussians=500000, log_interval=500, max_res=1600, output_name="quick_test")


In [ ]:
#@title === 7B: Full Training 30K (~30 min, single run) ===
from scripts.colab_pipeline import train_3dgs
train_3dgs(session, iterations=30000, max_gaussians=500000, log_interval=1000, max_res=1600, output_name="arena_3dgs")


---
## Step 8: Export & Validate for Unity (~1 min)
---


In [ ]:
#@title === 8A: Export Point Cloud (~1 min) ===
from scripts.colab_pipeline import export_pointcloud
export_pointcloud(session)


In [ ]:
#@title === 8B: Validate PLY for Unity (~1 min) ===
from scripts.colab_pipeline import validate_pointcloud
import os
validate_pointcloud("/content/arena_3dgs_pointcloud.ply")

print("\n" + "=" * 50)
print("  VIEWING OPTIONS")
print("=" * 50)
print("\n1. SuperSplat (no install, web):")
print("     https://supersplat.com/")
print("\n2. Unity walkthrough (best):")
print("     Clone: https://github.com/aras-p/UnityGaussianSplatting")
print("     Drop PLY into Assets/GaussianAssets/")
print("     WASD + mouse-look controls")
print("\n3. Local decompression viewer (lightweight, no GPU):")
print("     python3 scripts/decompress_splat.py compressed.splat")
print("     Drag to orbit, scroll to zoom, R=reset, Q=quit")


In [ ]:
#@title === 8C: Compress Model for Local Viewer (~1 min) ===
import urllib.request
import subprocess, sys, os

if session.is_step_done("compressed"):
    print("Compression already done (session state). Skipping.")
else:
    PLY_PATH = "/content/arena_3dgs_pointcloud.ply"
    if not os.path.exists(PLY_PATH):
        print("No PLY found. Run 8A first.")
    else:
        SCRIPT = "/content/compress_splat.py"
        if not os.path.exists(SCRIPT):
            url = "https://raw.githubusercontent.com/kaarthik-balakrishnan/arena-3dgs/main/scripts/compress_splat.py"
            urllib.request.urlretrieve(url, SCRIPT)
            print("Downloaded compress_splat.py")

        quality = session.get_param("compress_quality", "medium")
        print(f"Running compression (quality={quality})...")
        result = subprocess.run(
            [sys.executable, SCRIPT, PLY_PATH, "--quality", quality, "--output-dir", "/content"],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
        else:
            import glob
            splats = glob.glob("/content/*.splat")
            if splats:
                splat = splats[-1]
                size_mb = os.path.getsize(splat) / (1024 * 1024)
                print(f"\nCompressed: {splat} ({size_mb:.1f} MB)")
                session.set_param("compressed_size_mb", round(size_mb, 1))
                print("\nDownload the .splat file to your computer, then view:")
                print("  python3 scripts/decompress_splat.py path/to/arena_3dgs_compressed.splat")
                from google.colab import files
                files.download(splat)

    session.mark_step("compressed")


---
## Appendix: Troubleshooting

| Problem | Solution |
|---------|----------|
| **train_3dgs_enhanced.py not found** | Re-run Cell 2 to download from GitHub |
| **Session disconnected** | Re-open notebook, run Cell 1, choose Continue, run remaining cells |
| **CUDA out of memory** | Reduce `max_gaussians` or `max_res` in training cell |
| **COLMAP produces 0 images** | Download pre-computed data (Step 5) |
| **Drive restore failed** | Check `MyDrive/arena_3dgs/` exists and has files |

### Session State

Progress is tracked in `MyDrive/arena_3dgs/session_state.json`. On resume,
intermediate data (images, database, sparse models) is restored from Drive.

### Checkpoint locations (in Google Drive):
- `MyDrive/arena_3dgs/session_state.json` - Session progress tracker
- `MyDrive/arena_3dgs/database.db` - COLMAP features + matches
- `MyDrive/arena_3dgs/sparse_model` - COLMAP reconstruction
- `output/arena_3dgs/arena_3dgs.ply` - Trained model PLY file
- `MyDrive/arena_3dgs/arena_3dgs_pointcloud.ply` - Final PLY export

---
